In [ ]:
# 导入所需模块
from langgraph.graph import StateGraph, END
import random  # 用于生成随机数
from typing import Dict, List, TypedDict

In [ ]:
# 定义猜数字游戏的状态
class GameState(TypedDict):
    player_name: str       # 玩家名称
    target_number: int     # 目标数字（需要猜的数）
    guesses: List[int]     # 所有猜测记录
    attempts: int          # 已尝试次数
    hint: str              # 提示信息
    lower_bound: int       # 猜测范围下限
    upper_bound: int       # 猜测范围上限

In [ ]:
# 初始化节点：设置游戏参数
def setup_node(state: GameState) -> GameState:
    """Initialize the game with a random target number"""
    state["player_name"] = f"Welcome, {state['player_name']}!"
    # 生成1到20之间的随机目标数字
    state["target_number"] = random.randint(1, 20)
    state["guesses"] = []        # 清空猜测记录
    state["attempts"] = 0        # 重置尝试次数
    state["hint"] = "Game started! Try to guess the number."
    state["lower_bound"] = 1     # 初始下限
    state["upper_bound"] = 20    # 初始上限
    print(f"{state['player_name']} The game has begun. I'm thinking of a number between 1 and 20.")
    return state

In [ ]:
# 猜测节点：基于当前范围生成一个智能猜测
def guess_node(state: GameState) -> GameState:
    """Generate a smarter guess based on previous hints"""
    
    # 从当前范围中排除已猜过的数字，生成可选列表
    possible_guesses = [i for i in range(state["lower_bound"], state["upper_bound"] + 1) if i not in state["guesses"]]
    if possible_guesses:
        # 从可选数字中随机选一个
        guess = random.choice(possible_guesses)
    else:
        # 如果没有可选数字，随机猜一个范围内的数
        guess = random.randint(state["lower_bound"], state["upper_bound"])
    
    # 记录本次猜测并增加尝试次数
    state["guesses"].append(guess)
    state["attempts"] += 1
    print(f"Attempt {state['attempts']}: Guessing {guess} (Current range: {state['lower_bound']}-{state['upper_bound']})")
    return state

In [ ]:
# 提示节点：根据猜测结果给出提示并缩小范围
def hint_node(state: GameState) -> GameState:
    """Here we provide a hint based on the last guess and update the bounds"""
    latest_guess = state["guesses"][-1]  # 获取最新的猜测
    target = state["target_number"]
    
    if latest_guess < target:
        # 猜小了，提高下限
        state["hint"] = f"The number {latest_guess} is too low. Try higher!"
        state["lower_bound"] = max(state["lower_bound"], latest_guess + 1)
        print(f"Hint: {state['hint']}")
        
    elif latest_guess > target:
        # 猜大了，降低上限
        state["hint"] = f"The number {latest_guess} is too high. Try lower!"
        state["upper_bound"] = min(state["upper_bound"], latest_guess - 1)
        print(f"Hint: {state['hint']}")
    else:
        # 猜对了！
        state["hint"] = f"Correct! You found the number {target} in {state['attempts']} attempts."
        print(f"Success! {state['hint']}")
    
    return state

In [ ]:
# 条件判断函数：决定是继续猜还是结束游戏
def should_continue(state: GameState) -> str:
    """Determine if we should continue guessing or end the game"""
    
    # There are 2 end conditions - either 7 is reached or the correct number is guessed
    # 两个结束条件：猜对了或者达到最大尝试次数7次
    
    latest_guess = state["guesses"][-1]
    if latest_guess == state["target_number"]:
        # 猜对了，游戏结束
        print(f"GAME OVER: Number found!")
        return "end"
    elif state["attempts"] >= 7:
        # 超过最大尝试次数，游戏结束
        print(f"GAME OVER: Maximum attempts reached! The number was {state['target_number']}")
        return "end"
    else:
        # 继续猜
        print(f"CONTINUING: {state['attempts']}/7 attempts used")
        return "continue"

In [ ]:
# 构建猜数字游戏的状态图
graph = StateGraph(GameState)

# 添加三个节点：初始化、猜测、提示
graph.add_node("setup", setup_node)
graph.add_node("guess", guess_node)
graph.add_node("hint_node", hint_node)  

# 定义边：初始化后进入猜测，猜测后进入提示
graph.add_edge("setup", "guess")
graph.add_edge("guess", "hint_node")  

# 条件边：提示后根据结果决定继续猜还是结束
graph.add_conditional_edges(
    "hint_node", 
    should_continue,
    {
        "continue": "guess",  # 继续猜测（循环）
        "end": END             # 游戏结束
    }
)

# 设置入口点并编译
graph.set_entry_point("setup")
app = graph.compile()

In [ ]:
# 可视化游戏流程图
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# 运行猜数字游戏
result = app.invoke({"player_name": "Student", "guesses": [], "attempts": 0, "lower_bound": 1, "upper_bound": 20})